In [1]:
#!/usr/bin/env python3
"""
HAWKEYE — TRAIN & EXPORT (Kaggle notebook script)
==================================================
Runs the proven V8 AML mule-detection pipeline (your AUC=0.9978 / F1≈0.95 setup)
and EXPORTS the artifacts needed by the HAWKEYE serving stack:

  /kaggle/working/hawkeye_artifacts/
    ├── lgb_model_m1_full.txt           # LightGBM booster (clean feats), full-data refit
    ├── lgb_model_m2_full.txt           # LightGBM booster (all feats),   full-data refit
    ├── feature_config.json             # feat_cols, feat_clean, blend weights, threshold
    ├── feature_stats.json              # per-feature mean/std/min/max for synthesis & scoring
    ├── account_feature_matrix.parquet  # final F matrix (incl. predictions for test) — small
    ├── oof_predictions.parquet         # OOF probas for analysis
    ├── train_metadata.json             # AUC/F1/threshold/blend/feature-counts
    └── submission.csv                  # original Kaggle submission (sanity)

Drop this file into a Kaggle notebook with the RBI NFPC Phase-2 dataset attached.
After it finishes, download the `hawkeye_artifacts/` folder and feed it to
the HAWKEYE backend (see Claude Code prompt).
"""
import pandas as pd, numpy as np, lightgbm as lgb
from glob import glob
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, f1_score,
                              precision_score, recall_score,
                              precision_recall_curve)
import gc, os, json, time, warnings, pickle
warnings.filterwarnings('ignore')

try:
    from tqdm.auto import tqdm
except ImportError:
    class tqdm:
        def __init__(self, it=None, **kw): self.it=it; self.total=kw.get('total',0)
        def __iter__(self):
            for i,x in enumerate(self.it):
                if (i+1)%20==0: print(f"  {i+1}/{self.total}")
                yield x

VERSION = "HAWKEYE_export_v1"
BASE = '/kaggle/input/datasets/abhyudayrbih/rbih-nfpc-phase-2'
if not os.path.exists(BASE): BASE = '/kaggle/input/rbih-nfpc-phase-2'
if not os.path.exists(BASE):
    for p in glob('/kaggle/input/*/'):
        if os.path.exists(f'{p}/train_labels.parquet'): BASE = p.rstrip('/'); break

OUTPUT     = '/kaggle/working'
ARTIFACTS  = f'{OUTPUT}/hawkeye_artifacts'
os.makedirs(ARTIFACTS, exist_ok=True)
SEED = 42; np.random.seed(SEED); N_FOLDS = 5

print("="*70); print(f"VERSION: {VERSION}"); print("="*70)
t0 = time.time()

# ══════════════════════════════════════════════════════════════════════════
# LOAD
# ══════════════════════════════════════════════════════════════════════════
train_labels    = pd.read_parquet(f'{BASE}/train_labels.parquet')
test_accounts   = pd.read_parquet(f'{BASE}/test_accounts.parquet')
accounts        = pd.read_parquet(f'{BASE}/accounts.parquet')
accounts_add    = pd.read_parquet(f'{BASE}/accounts-additional.parquet')
customers       = pd.read_parquet(f'{BASE}/customers.parquet')
demographics    = pd.read_parquet(f'{BASE}/demographics.parquet')
product_details = pd.read_parquet(f'{BASE}/product_details.parquet')
branch_df       = pd.read_parquet(f'{BASE}/branch.parquet')
linkage         = pd.read_parquet(f'{BASE}/customer_account_linkage.parquet')

target_ids     = set(train_labels['account_id']) | set(test_accounts['account_id'])
train_acct_set = set(train_labels['account_id'])
test_acct_set  = set(test_accounts['account_id'])
mule_ids       = set(train_labels[train_labels['is_mule']==1]['account_id'])
print(f"Train: {len(train_labels):,} | Test: {len(test_accounts):,} | Mules: {len(mule_ids):,}")

def tid_int(s):
    try: return int(s[4:])
    except: return -1

# ══════════════════════════════════════════════════════════════════════════
# [1/9] MAIN TRANSACTION LOOP
# ══════════════════════════════════════════════════════════════════════════
txn_parts = sorted(glob(f'{BASE}/transactions/batch-*/part_*.parquet'))
print(f"\n[1/9] Main transaction loop ({len(txn_parts)} parts)...")
t1 = time.time()

p_edges, p_stats, p_monthly = [], [], []
txn_range_index = []
BATCH = 20

for i in tqdm(range(len(txn_parts)), desc="Txn", total=len(txn_parts)):
    df = pd.read_parquet(txn_parts[i], columns=[
        'transaction_id','account_id','transaction_timestamp',
        'amount','txn_type','counterparty_id','mcc_code','channel'])

    all_ids = df['transaction_id']
    txn_range_index.append((tid_int(all_ids.min()), tid_int(all_ids.max()), txn_parts[i]))

    edge = df.groupby(['account_id','counterparty_id'], sort=False).agg(
        ew=('amount', lambda x: x.abs().sum()), ec=('amount','count')
    ).reset_index()
    p_edges.append(edge)

    tdf = df[df['account_id'].isin(target_ids)].copy()
    if len(tdf) > 0:
        tdf['aa']  = tdf['amount'].abs()
        tdf['ts']  = pd.to_datetime(tdf['transaction_timestamp'], format='ISO8601')
        tdf['hr']  = tdf['ts'].dt.hour
        tdf['dw']  = tdf['ts'].dt.dayofweek
        tdf['dom'] = tdf['ts'].dt.day
        tdf['ic']  = (tdf['txn_type']=='C').astype(np.int8)
        tdf['id_'] = (tdf['txn_type']=='D').astype(np.int8)
        tdf['ca']  = tdf['aa']*tdf['ic']; tdf['da']  = tdf['aa']*tdf['id_']
        tdf['sq']  = tdf['aa']**2; tdf['dt']  = tdf['ts'].dt.date
        for amt,col in [(1000,'r1k'),(5000,'r5k'),(10000,'r10k'),(25000,'r25k'),(50000,'r50k')]:
            tdf[col] = (tdf['aa']==amt).astype(np.int8)
        tdf['s45'] = ((tdf['aa']>=45000)&(tdf['aa']<50000)).astype(np.int8)
        tdf['s49'] = ((tdf['aa']>=49000)&(tdf['aa']<50000)).astype(np.int8)
        tdf['ngt'] = ((tdf['hr']<6)|(tdf['hr']>=23)).astype(np.int8)
        tdf['wkd'] = (tdf['dw']>=5).astype(np.int8)
        tdf['biz'] = ((tdf['hr']>=9)&(tdf['hr']<=17)&(tdf['dw']<5)).astype(np.int8)
        tdf['xlg'] = (tdf['aa']>=100000).astype(np.int8)
        tdf['lrg'] = ((tdf['aa']>=10000)&(tdf['aa']<100000)).astype(np.int8)
        tdf['med'] = ((tdf['aa']>=1000)&(tdf['aa']<10000)).astype(np.int8)
        tdf['sml'] = ((tdf['aa']>=100)&(tdf['aa']<1000)).astype(np.int8)
        tdf['mic'] = (tdf['aa']<100).astype(np.int8)
        tdf['mbd'] = tdf['dom'].isin([1,2,3,28,29,30,31]).astype(np.int8)
        tdf['mk']  = tdf['ts'].dt.to_period('M').astype(str)

        st = tdf.groupby('account_id', sort=False).agg(
            n=('aa','count'), sa=('aa','sum'), sq=('sq','sum'),
            mx=('aa','max'), mn=('aa','min'),
            sc=('ca','sum'), sd=('da','sum'), nc=('ic','sum'),
            r1k=('r1k','sum'),r5k=('r5k','sum'),r10k=('r10k','sum'),
            r25k=('r25k','sum'),r50k=('r50k','sum'),
            s45=('s45','sum'),s49=('s49','sum'),
            ngt=('ngt','sum'),wkd=('wkd','sum'),biz=('biz','sum'),
            xlg=('xlg','sum'),lrg=('lrg','sum'),med=('med','sum'),
            sml=('sml','sum'),mic=('mic','sum'),mbd=('mbd','sum'),
            tmin=('ts','min'),tmax=('ts','max'),
            nch=('channel','nunique'),nmcc=('mcc_code','nunique'),
            ndays=('dt','nunique'),ncp=('counterparty_id','nunique'),
            n_months=('mk','nunique'),
        ).reset_index()
        fan_c = (tdf[tdf['ic']==1].groupby('account_id')['counterparty_id']
                 .nunique().reset_index().rename(columns={'counterparty_id':'ncp_c'}))
        fan_d = (tdf[tdf['id_']==1].groupby('account_id')['counterparty_id']
                 .nunique().reset_index().rename(columns={'counterparty_id':'ncp_d'}))
        st = st.merge(fan_c, on='account_id', how='left').merge(fan_d, on='account_id', how='left')
        st['ncp_c'] = st['ncp_c'].fillna(0).astype(np.int32)
        st['ncp_d'] = st['ncp_d'].fillna(0).astype(np.int32)
        p_stats.append(st)

        mo = tdf.groupby(['account_id','mk'], sort=False).agg(
            mc=('aa','count'), ma=('aa','sum'),
            mcr=('ca','sum'), mdr=('da','sum'),
            mcp=('counterparty_id','nunique'),
            mr50=('r50k','sum'), mxlg=('xlg','sum'),
            m_tmin=('ts','min'), m_tmax=('ts','max'),
        ).reset_index()
        p_monthly.append(mo)

    del df, tdf, edge; gc.collect()

    if (i+1)%BATCH==0 or (i+1)==len(txn_parts):
        if len(p_edges)>1:
            c=pd.concat(p_edges,ignore_index=True)
            p_edges=[c.groupby(['account_id','counterparty_id'],sort=False).agg(
                ew=('ew','sum'),ec=('ec','sum')).reset_index()]
            del c; gc.collect()
        if len(p_stats)>1:
            c=pd.concat(p_stats,ignore_index=True)
            p_stats=[c.groupby('account_id',sort=False).agg(
                n=('n','sum'),sa=('sa','sum'),sq=('sq','sum'),
                mx=('mx','max'),mn=('mn','min'),
                sc=('sc','sum'),sd=('sd','sum'),nc=('nc','sum'),
                r1k=('r1k','sum'),r5k=('r5k','sum'),r10k=('r10k','sum'),
                r25k=('r25k','sum'),r50k=('r50k','sum'),
                s45=('s45','sum'),s49=('s49','sum'),
                ngt=('ngt','sum'),wkd=('wkd','sum'),biz=('biz','sum'),
                xlg=('xlg','sum'),lrg=('lrg','sum'),med=('med','sum'),
                sml=('sml','sum'),mic=('mic','sum'),mbd=('mbd','sum'),
                tmin=('tmin','min'),tmax=('tmax','max'),
                nch=('nch','max'),nmcc=('nmcc','max'),
                ndays=('ndays','max'),ncp=('ncp','max'),
                ncp_c=('ncp_c','max'),ncp_d=('ncp_d','max'),
                n_months=('n_months','max'),
            ).reset_index()]
            del c; gc.collect()
        if len(p_monthly)>1:
            c=pd.concat(p_monthly,ignore_index=True)
            p_monthly=[c.groupby(['account_id','mk'],sort=False).agg(
                mc=('mc','sum'),ma=('ma','sum'),
                mcr=('mcr','sum'),mdr=('mdr','sum'),
                mcp=('mcp','sum'),mr50=('mr50','sum'),mxlg=('mxlg','sum'),
                m_tmin=('m_tmin','min'),m_tmax=('m_tmax','max'),
            ).reset_index()]
            del c; gc.collect()

edges=p_edges[0]; S=p_stats[0]; monthly_all=p_monthly[0]
del p_edges,p_stats,p_monthly; gc.collect()
print(f"  Done {time.time()-t1:.0f}s | Edges: {len(edges):,} | Stats: {len(S):,}")

# ══════════════════════════════════════════════════════════════════════════
# [2/9] TADD: BALANCE + IP — RANGE-BASED LOADING
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[2/9] Loading tadd (balance + IP) via range-based matching...")
t2 = time.time()
all_tadd = sorted(glob(f'{BASE}/transactions_additional/batch-*/part_*.parquet'))
p_bal, p_ip = [], []
tadd_covered_accounts = set()

for tadd_p in tqdm(all_tadd, desc="tadd", total=len(all_tadd)):
    try:
        tadd_ids_df = pd.read_parquet(tadd_p, columns=['transaction_id'])
        tadd_mn = tid_int(tadd_ids_df['transaction_id'].min())
        tadd_mx = tid_int(tadd_ids_df['transaction_id'].max())
        tadd_id_set = set(tadd_ids_df['transaction_id'])
        del tadd_ids_df; gc.collect()
        overlapping_txn = [(mn,mx,p) for mn,mx,p in txn_range_index
                            if mn<=tadd_mx and mx>=tadd_mn]
        if not overlapping_txn: continue
        txn_frames = []
        for mn,mx,txn_p in overlapping_txn:
            df = pd.read_parquet(txn_p, columns=['transaction_id','account_id'])
            df = df[df['account_id'].isin(target_ids) & df['transaction_id'].isin(tadd_id_set)]
            if len(df)>0: txn_frames.append(df)
            del df; gc.collect()
        if not txn_frames:
            del tadd_id_set; gc.collect(); continue
        txn_combined = pd.concat(txn_frames, ignore_index=True)
        del txn_frames, tadd_id_set; gc.collect()
        tadd = pd.read_parquet(tadd_p, columns=['transaction_id','balance_after_transaction','ip_address'])
        merged = txn_combined.merge(tadd, on='transaction_id', how='inner')
        del txn_combined, tadd; gc.collect()
        if len(merged)==0: continue
        tadd_covered_accounts.update(merged['account_id'].unique())
        merged['bat'] = merged['balance_after_transaction'].astype(float)
        merged['nz']  = (merged['bat'].abs() < 5000).astype(np.int8)
        bal_agg = merged.groupby('account_id').agg(
            bal_sum=('bat','sum'), bal_sum_sq=('bat', lambda x:(x**2).sum()),
            bal_min=('bat','min'), bal_max=('bat','max'),
            bal_n=('bat','count'), bal_nz=('nz','sum')).reset_index()
        p_bal.append(bal_agg)
        ip_df = merged[merged['ip_address'].notna()][['account_id','ip_address']].drop_duplicates()
        if len(ip_df)>0: p_ip.append(ip_df)
        del merged, bal_agg, ip_df; gc.collect()
    except Exception as e:
        print(f"  WARN {os.path.basename(tadd_p)}: {e}"); continue

print(f"  Covered accounts: {len(tadd_covered_accounts):,} | Mule cov: "
      f"{len(tadd_covered_accounts & mule_ids):,}/{len(mule_ids):,}")

if p_bal:
    c = pd.concat(p_bal, ignore_index=True)
    bal_raw = c.groupby('account_id', sort=False).agg(
        bal_sum=('bal_sum','sum'), bal_sum_sq=('bal_sum_sq','sum'),
        bal_min=('bal_min','min'), bal_max=('bal_max','max'),
        bal_n=('bal_n','sum'), bal_nz=('bal_nz','sum')).reset_index()
    del c, p_bal; gc.collect()
    bal_raw['bal_mean']  = bal_raw['bal_sum'] / bal_raw['bal_n'].clip(lower=1)
    bal_raw['bal_var']   = (bal_raw['bal_sum_sq'] / bal_raw['bal_n'].clip(lower=1) - bal_raw['bal_mean']**2)
    bal_raw['bal_std']   = np.sqrt(bal_raw['bal_var'].clip(lower=0))
    bal_raw['bal_range'] = bal_raw['bal_max'] - bal_raw['bal_min']
    bal_raw['pass_rate'] = bal_raw['bal_nz'] / bal_raw['bal_n'].clip(lower=1)
    bal_raw['bal_cv']    = bal_raw['bal_std'] / (bal_raw['bal_mean'].abs()+1)
    bal_feats = bal_raw[['account_id','bal_mean','bal_std','bal_min','bal_max',
                          'bal_range','bal_n','pass_rate','bal_cv']].copy()
    del bal_raw; gc.collect()
else:
    bal_feats = pd.DataFrame(columns=['account_id','bal_mean','bal_std','bal_min',
                                       'bal_max','bal_range','bal_n','pass_rate','bal_cv'])

if p_ip:
    ip_all = pd.concat(p_ip, ignore_index=True).drop_duplicates()
    del p_ip; gc.collect()
    mule_ip_global = set(ip_all[ip_all['account_id'].isin(mule_ids)]['ip_address'])
    n_unique_ips = (ip_all.groupby('account_id')['ip_address']
                   .nunique().reset_index().rename(columns={'ip_address':'n_unique_ips'}))
    use_ip = ip_all['account_id'].nunique() > 10000
else:
    ip_all = pd.DataFrame(columns=['account_id','ip_address'])
    mule_ip_global = set()
    n_unique_ips = pd.DataFrame(columns=['account_id','n_unique_ips'])
    use_ip = False
print(f"  IP enabled: {use_ip} | tadd pass: {time.time()-t2:.0f}s")

# ══════════════════════════════════════════════════════════════════════════
# [3/9] GRAPH FEATURES (V4-style)
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[3/9] Graph features...")
target_edges = edges[edges['account_id'].isin(target_ids)].copy()
gf0 = target_edges.groupby('account_id').agg(
    g_ncp=('counterparty_id','nunique'),g_tew=('ew','sum'),
    g_maxew=('ew','max'),g_nec=('ec','sum'),
    g_medew=('ew','median'),g_stdew=('ew','std')).reset_index()
gf0['g_top1'] = gf0['g_maxew']/(gf0['g_tew']+1)
gf0['g_cvew'] = gf0['g_stdew']/(gf0['g_medew']+1)
gf0['g_avg_txn_per_cp'] = gf0['g_nec']/(gf0['g_ncp']+1)
t3 = target_edges.groupby('account_id')['ew'].apply(lambda x: x.nlargest(3).sum()).reset_index()
t3.columns = ['account_id','g_t3']
gf0 = gf0.merge(t3, on='account_id', how='left')
gf0['g_top3'] = gf0['g_t3']/(gf0['g_tew']+1)
hhi = target_edges.copy()
hhi['share'] = hhi['ew'] / hhi.groupby('account_id')['ew'].transform('sum')
hhi['share_sq'] = hhi['share']**2
hhi_s = hhi.groupby('account_id')['share_sq'].sum().reset_index()
hhi_s.columns = ['account_id','g_hhi']
gf0 = gf0.merge(hhi_s, on='account_id', how='left')
gf0.drop(columns=['g_t3','g_maxew','g_stdew','g_medew'], inplace=True)
del t3, hhi, hhi_s; gc.collect()

skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
train_aids = train_labels['account_id'].values
train_y    = train_labels['is_mule'].values
labeled_edges = edges[edges['account_id'].isin(train_acct_set)].merge(
    train_labels[['account_id','is_mule']], on='account_id', how='inner')

def compute_cp_stats(le):
    cp = le.groupby('counterparty_id').agg(nm=('is_mule','sum'),nl=('is_mule','count')).reset_index()
    cp['cp_mr'] = cp['nm']/cp['nl']
    cp['cp_excl'] = ((cp['nl']==cp['nm']) & (cp['nm']>0)).astype(int)
    mu = le[le['is_mule']==1].groupby('counterparty_id')['account_id'].nunique().reset_index()
    mu.columns = ['counterparty_id','cp_n_mule_users']
    return cp.merge(mu, on='counterparty_id', how='left').fillna({'cp_n_mule_users':0})

def compute_graph_feats(acct_edges, cp_st):
    ae = acct_edges.merge(cp_st[['counterparty_id','cp_mr','cp_excl','cp_n_mule_users']],
                           on='counterparty_id', how='left'
                           ).fillna({'cp_mr':0,'cp_excl':0,'cp_n_mule_users':0})
    ae['wms'] = ae['cp_mr'] * ae['ew']
    gf = ae.groupby('account_id').agg(
        g_mcs=('cp_mr','sum'), g_mcm=('cp_mr','mean'),
        g_mcx=('cp_mr','max'), g_wms=('wms','sum'),
        g_nexcl=('cp_excl','sum'),
        g_mule_users_sum=('cp_n_mule_users','sum'),
        g_mule_users_max=('cp_n_mule_users','max')).reset_index()
    tew = ae.groupby('account_id')['ew'].sum().reset_index().rename(columns={'ew':'_tw'})
    gf = gf.merge(tew, on='account_id', how='left')
    gf['g_wmsn'] = gf['g_wms']/(gf['_tw']+1)
    nc = ae.groupby('account_id').size().reset_index(name='_ncp')
    gf = gf.merge(nc, on='account_id', how='left')
    gf['g_pexcl'] = gf['g_nexcl']/(gf['_ncp']+1)
    gf.drop(columns=['_tw','_ncp'], inplace=True)
    for t in [0.05, 0.1, 0.3, 0.5]:
        col = f'g_gt{int(t*100)}'
        tmp = ae[ae['cp_mr']>t].groupby('account_id').size().reset_index(name=col)
        gf = gf.merge(tmp, on='account_id', how='left'); gf[col] = gf[col].fillna(0)
    return gf

cv_graph_parts = []
ip_cv_parts    = []
for fold,(ti,vi) in enumerate(skf.split(train_aids, train_y)):
    tr_set = set(train_aids[ti]); va_set = set(train_aids[vi])
    fold_cp = compute_cp_stats(labeled_edges[labeled_edges['account_id'].isin(tr_set)])
    cv_graph_parts.append(compute_graph_feats(target_edges[target_edges['account_id'].isin(va_set)], fold_cp))
    if use_ip:
        fold_mule_ips = set(ip_all[ip_all['account_id'].isin(tr_set & mule_ids)]['ip_address'])
        va_ip = ip_all[ip_all['account_id'].isin(va_set)].copy()
        va_ip['is_mule_ip'] = va_ip['ip_address'].isin(fold_mule_ips).astype(int)
        ip_fold = va_ip.groupby('account_id').agg(
            ip_mule_shared=('is_mule_ip','sum'),
            ip_has_mule_ip=('is_mule_ip','max')).reset_index()
        ip_cv_parts.append(ip_fold)

full_cp = compute_cp_stats(labeled_edges)
test_gf = compute_graph_feats(target_edges[target_edges['account_id'].isin(test_acct_set)], full_cp)
graph_cv = pd.concat(cv_graph_parts + [test_gf], ignore_index=True)

if use_ip:
    test_ip = ip_all[ip_all['account_id'].isin(test_acct_set)].copy()
    test_ip['is_mule_ip'] = test_ip['ip_address'].isin(mule_ip_global).astype(int)
    test_ip_feats = test_ip.groupby('account_id').agg(
        ip_mule_shared=('is_mule_ip','sum'),
        ip_has_mule_ip=('is_mule_ip','max')).reset_index()
    ip_feats = pd.concat(ip_cv_parts + [test_ip_feats], ignore_index=True)
    ip_feats = ip_feats.merge(n_unique_ips, on='account_id', how='left')
    ip_feats['ip_mule_rate'] = ip_feats['ip_mule_shared']/ip_feats['n_unique_ips'].clip(lower=1)

del cv_graph_parts, test_gf, labeled_edges, target_edges, full_cp, ip_all; gc.collect()
print(f"  Graph: {graph_cv.shape}")

# ══════════════════════════════════════════════════════════════════════════
# [4/9] BURSTINESS
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[4/9] Burstiness...")
def extract_burstiness(monthly_df):
    feats = []
    for aid, grp in monthly_df.groupby('account_id'):
        grp = grp.sort_values('mk').reset_index(drop=True)
        vols = grp['ma'].values.astype(float); n_mo = len(grp)
        mu = vols.mean()+1e-9; mx = vols.max(); sv = np.sort(vols); n = len(sv)
        gini = max(0., 2*np.sum(np.arange(1,n+1)*sv)/(n*sv.sum()+1e-9) - (n+1)/n)
        peak_idx = int(np.argmax(vols)); b_months = int((vols>2*mu).sum())
        cps = grp['mcp'].values.astype(float)
        mom_max = float(np.diff(vols).max()/mu) if n_mo>1 else 0.
        feats.append({'account_id':aid,
            'b_max_vol':float(mx),'b_spike_ratio':float(mx/mu),
            'b_gini':float(gini),'b_cv':float(vols.std()/mu),
            'b_burst_months':b_months,'b_burst_frac':b_months/max(n_mo,1),
            'b_peak_recency':peak_idx/max(n_mo-1,1),
            'b_n_months':n_mo,'b_max_cps':float(cps.max()),
            'b_cp_spike':float(cps.max()/(cps.mean()+1)),
            'b_max_r50k':int(grp['mr50'].max()),'b_max_xlg':int(grp['mxlg'].max()),
            'b_mom_max_jump':mom_max,'b_active_frac':n_mo/60.})
    return pd.DataFrame(feats)
burst_feats = extract_burstiness(monthly_all)

# ══════════════════════════════════════════════════════════════════════════
# [5/9] BEHAVIORAL FEATURES
# ══════════════════════════════════════════════════════════════════════════
ts_df = S[['account_id','tmin','tmax']].copy()
n = S['n'].clip(lower=1)
S['mean_amt'] = S['sa']/n
S['std_amt']  = np.sqrt(np.maximum(0, S['sq']/n - (S['sa']/n)**2))
S['cv_amt']   = S['std_amt']/(S['mean_amt']+1)
S['cr']       = S['nc']/n; S['cdr'] = S['sc']/(S['sd']+1)
S['nfr']      = (S['sc']-S['sd']).abs()/(S['sa']+1)
S['pt']       = np.minimum(S['sc'],S['sd'])/(np.maximum(S['sc'],S['sd'])+1)
S['span']     = (S['tmax']-S['tmin']).dt.days.clip(lower=1)
S['tpd']      = S['n']/S['ndays'].clip(lower=1); S['dcov'] = S['ndays']/S['span']
S['cppt']     = S['ncp']/n; S['amt_range'] = S['mx']-S['mn']
for r in ['1k','5k','10k','25k','50k']: S[f'pr{r}'] = S[f'r{r}']/n
S['trp']  = (S['r1k']+S['r5k']+S['r10k']+S['r25k']+S['r50k'])/n
S['hrs']  = S['pr10k']*2+S['pr25k']*5+S['pr50k']*10
S['ps45'] = S['s45']/n; S['ps49'] = S['s49']/n
S['pngt'] = S['ngt']/n; S['pwkd'] = S['wkd']/n; S['pbiz'] = S['biz']/n
for b in ['xlg','lrg','med','sml','mic']: S[f'p{b}'] = S[b]/n
S['txn_per_month'] = S['n']/S['n_months'].clip(lower=1)
S['amt_per_month'] = S['sa']/S['n_months'].clip(lower=1)
S['fan_ratio']  = S['ncp_c']/(S['ncp_d']+1)
S['fan_asymm']  = (S['ncp_c']-S['ncp_d']).abs()/(S['ncp_c']+S['ncp_d']+1)
S['p_mbd']      = S['mbd']/n
S.drop(columns=[c for c in ['sq','tmin','tmax','r1k','r5k','r10k','r25k','r50k',
    's45','s49','ngt','wkd','biz','xlg','lrg','med','sml','mic','mn','mbd',
    'ncp_c','ncp_d'] if c in S.columns], inplace=True)

# ══════════════════════════════════════════════════════════════════════════
# [6/9] STATIC FEATURES
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[6/9] Static features...")
ref = pd.Timestamp('2025-06-30')
af = accounts[['account_id','product_code','avg_balance','product_family',
    'num_chequebooks','kyc_compliant','rural_branch',
    'monthly_avg_balance','quarterly_avg_balance','daily_avg_balance',
    'account_opening_date','last_kyc_date','branch_code',
    'nomination_flag','cheque_allowed','cheque_availed',
    'last_mobile_update_date']].copy()
af['pf']    = af['product_family'].map({'S':0,'K':1,'O':2})
af['kyc_e'] = (af['kyc_compliant']=='Y').astype(int)
af['rur']   = (af['rural_branch']=='Y').astype(int)
af['nom']   = (af['nomination_flag']=='Y').astype(int)
af['chk_a'] = (af['cheque_allowed']=='Y').astype(int)
af['chk_v'] = (af['cheque_availed']=='Y').astype(int)
af['odt']   = pd.to_datetime(af['account_opening_date'], errors='coerce')
af['age_d'] = (ref-af['odt']).dt.days
af['kdt']   = pd.to_datetime(af['last_kyc_date'], errors='coerce')
af['kyc_d'] = (ref-af['kdt']).dt.days
af['bmd']   = af['monthly_avg_balance']/(af['daily_avg_balance'].abs()+1)
af['bqd']   = af['quarterly_avg_balance']/(af['daily_avg_balance'].abs()+1)
af['mob_dt']= pd.to_datetime(af['last_mobile_update_date'], errors='coerce')
af['has_mob_change']  = af['mob_dt'].notna().astype(int)
af['mob_change_days'] = (ref-af['mob_dt']).dt.days
af.drop(columns=['product_family','kyc_compliant','rural_branch',
    'account_opening_date','last_kyc_date','odt','kdt','nomination_flag',
    'cheque_allowed','cheque_availed','last_mobile_update_date','mob_dt'], inplace=True)
sch = accounts_add.copy()
sch['pmjdy']   = (sch['scheme_code']=='PMJDY').astype(int)
sch['regular'] = (sch['scheme_code']=='REGULAR').astype(int)
sch = sch[['account_id','pmjdy','regular']]
cf = linkage.merge(customers[['customer_id','date_of_birth','relationship_start_date',
    'pan_available','mobile_banking_flag','internet_banking_flag',
    'atm_card_flag','credit_card_flag','demat_flag','fastag_flag']],
    on='customer_id', how='left')
cf['dob']   = pd.to_datetime(cf['date_of_birth'], errors='coerce')
cf['age']   = (ref-cf['dob']).dt.days/365.25
cf['rs']    = pd.to_datetime(cf['relationship_start_date'], errors='coerce')
cf['rel_y'] = (ref-cf['rs']).dt.days/365.25
for c in ['pan_available','mobile_banking_flag','internet_banking_flag',
           'atm_card_flag','credit_card_flag','demat_flag','fastag_flag']:
    cf[c[:3]] = (cf[c]=='Y').astype(int)
cf['ndig'] = cf['pan']+cf['mob']+cf['int']+cf['atm']+cf['cre']+cf['dem']+cf['fas']
cf = cf[['account_id','age','rel_y','ndig','pan','mob','int','atm','cre','dem','fas']]
df_ = linkage.merge(demographics[['customer_id','gender','nri_flag','joint_account_flag']],
    on='customer_id', how='left')
df_['male'] = (df_['gender']=='M').astype(int)
df_['nri']  = (df_['nri_flag']=='Y').astype(int)
df_['jnt']  = (df_['joint_account_flag']=='Y').astype(int)
df_ = df_[['account_id','male','nri','jnt']]
pf_df = linkage.merge(product_details, on='customer_id', how='left')
pf_df = pf_df[['account_id','loan_sum','loan_count','cc_sum','cc_count',
              'od_sum','od_count','ka_sum','ka_count','sa_sum','sa_count']]
bf_br = branch_df[['branch_code','branch_employee_count','branch_turnover',
                   'branch_asset_size','branch_type']].copy()
bf_br['bt'] = bf_br['branch_type'].map({'urban':0,'semi-urban':1,'rural':2})
bf_br.drop(columns=['branch_type'], inplace=True)
na = linkage.groupby('customer_id')['account_id'].count().reset_index()
na.columns = ['customer_id','nacct']
ma = linkage.merge(na, on='customer_id', how='left')[['account_id','nacct']]

# ══════════════════════════════════════════════════════════════════════════
# [7/9] MERGE
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[7/9] Merging...")
ids = pd.concat([train_labels[['account_id']], test_accounts[['account_id']]]).drop_duplicates()
F = ids.copy()
merge_list = [gf0, graph_cv, S, burst_feats, af, sch, cf, df_, pf_df, ma, bal_feats]
if use_ip: merge_list.append(ip_feats)
for d in merge_list: F = F.merge(d, on='account_id', how='left')
bc = accounts[['account_id','branch_code']].drop_duplicates()
F = F.merge(bc, on='account_id', how='left', suffixes=('','_d'))
if 'branch_code_d' in F.columns: F.drop(columns=['branch_code_d'], inplace=True)
F = F.merge(bf_br, on='branch_code', how='left')
F = F.merge(train_labels[['account_id','is_mule']], on='account_id', how='left')
F['tpb']  = F['n']/(F['avg_balance'].abs()+1)
F['atb']  = F['mean_amt']/(F['avg_balance'].abs()+1)
F['thru'] = F['sa']/F['span'].clip(lower=1)
F['g_mcs_per_cp'] = F['g_mcs']/(F['g_ncp']+1)
F['new_acct_high_vol'] = F['sa']/(F['age_d'].clip(lower=1)**0.5)
F['mob_change_recent'] = (F['mob_change_days'].fillna(9999)<365).astype(int)
del gf0, graph_cv, S, burst_feats, af, sch, cf, df_, pf_df, ma, bc, bf_br, bal_feats
gc.collect()
print(f"  Feature matrix: {F.shape}")

# ══════════════════════════════════════════════════════════════════════════
# [8/9] TRAINING (V8 hyperparameters, identical)
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[8/9] Training...")
exc = {'account_id','is_mule','branch_code',
       'freeze_date','unfreeze_date','has_freeze','frozen','account_status'}
feat_cols = [c for c in F.columns if c not in exc]
tm = F['is_mule'].notna()
Xall = F[feat_cols].values.astype(np.float32)

# Adversarial validation
adv_m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05,
                            num_leaves=63, random_state=SEED, verbose=-1)
adv_m.fit(Xall, tm.astype(int).values)
adv_imp = (pd.DataFrame({'f':feat_cols,'imp':adv_m.feature_importances_})
            .assign(imp_norm=lambda x: x['imp']/(x['imp'].sum()+1))
            .sort_values('imp', ascending=False))
rh = adv_imp[(adv_imp['imp_norm']>0.01) &
             (~adv_imp['f'].str.startswith('g_')) &
             (~adv_imp['f'].isin(['sa','n','ndays','mean_amt']))]['f'].tolist()
rh = list(set(rh) | {f for f in ['account_status','freeze_date','unfreeze_date',
                                   'frozen','has_freeze'] if f in feat_cols})
print(f"  Excluding {len(rh)}: {rh}")
feat_clean = [c for c in feat_cols if c not in rh]

Xt   = F.loc[tm,  feat_cols ].values.astype(np.float32)
yt   = F.loc[tm, 'is_mule'  ].values.astype(int)
Xte  = F.loc[~tm, feat_cols ].values.astype(np.float32)
tids = F.loc[~tm, 'account_id'].values
X1t  = F.loc[tm,  feat_clean].values.astype(np.float32)
X1e  = F.loc[~tm, feat_clean].values.astype(np.float32)

# Confident learning
print("  Noise detection pass...")
oof_noise = np.zeros(len(yt))
for fold,(ti,vi) in enumerate(StratifiedKFold(5,shuffle=True,random_state=SEED).split(Xt,yt)):
    spw = (yt[ti]==0).sum()/(yt[ti]==1).sum()
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31,
                            scale_pos_weight=spw, random_state=SEED, verbose=-1)
    m.fit(Xt[ti], yt[ti], eval_set=[(Xt[vi], yt[vi])],
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    oof_noise[vi] = m.predict_proba(Xt[vi])[:,1]
noise = np.zeros(len(yt), dtype=bool)
noise[(yt==1) & (oof_noise<0.10)] = True
noise[(yt==0) & (oof_noise>0.90)] = True
weights = np.ones(len(yt)); weights[noise] = 0.3
print(f"  Noisy: {noise.sum()}")

skf_train = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)

# Model 1
print("\n  --- Model 1 (clean feats) ---")
p1 = dict(objective='binary', metric='auc', n_estimators=5000, learning_rate=0.005,
          num_leaves=127, max_depth=10, min_child_samples=50,
          subsample=0.8, colsample_bytree=0.6,
          reg_alpha=0.05, reg_lambda=0.5, is_unbalance=True,
          random_state=SEED, n_jobs=-1, verbose=-1)
oof1 = np.zeros(len(yt)); tst1 = np.zeros(len(X1e))
m1_iters = []
for fold,(ti,vi) in enumerate(skf_train.split(X1t,yt)):
    m = lgb.LGBMClassifier(**p1)
    m.fit(X1t[ti], yt[ti], sample_weight=weights[ti],
          eval_set=[(X1t[vi], yt[vi])],
          callbacks=[lgb.early_stopping(300), lgb.log_evaluation(500)])
    oof1[vi] = m.predict_proba(X1t[vi])[:,1]
    tst1 += m.predict_proba(X1e)[:,1]/N_FOLDS
    m1_iters.append(m.best_iteration_)
    print(f"  M1 F{fold+1}: AUC={roc_auc_score(yt[vi],oof1[vi]):.5f} iters={m.best_iteration_}")
m1_oof_auc = roc_auc_score(yt, oof1)
print(f"  M1 OOF: {m1_oof_auc:.6f}")

# Model 2
print("\n  --- Model 2 (all feats) ---")
p2 = dict(objective='binary', metric='auc', n_estimators=4000, learning_rate=0.007,
          num_leaves=63, max_depth=8, min_child_samples=50,
          subsample=0.85, colsample_bytree=0.7,
          reg_alpha=0.1, reg_lambda=1.0,
          scale_pos_weight=(yt==0).sum()/(yt==1).sum(),
          random_state=SEED+1, n_jobs=-1, verbose=-1)
oof2 = np.zeros(len(yt)); tst2 = np.zeros(len(Xte))
m2_iters = []
for fold,(ti,vi) in enumerate(skf_train.split(Xt,yt)):
    m = lgb.LGBMClassifier(**p2)
    m.fit(Xt[ti], yt[ti],
          eval_set=[(Xt[vi], yt[vi])],
          callbacks=[lgb.early_stopping(300), lgb.log_evaluation(500)])
    oof2[vi] = m.predict_proba(Xt[vi])[:,1]
    tst2 += m.predict_proba(Xte)[:,1]/N_FOLDS
    m2_iters.append(m.best_iteration_)
    print(f"  M2 F{fold+1}: AUC={roc_auc_score(yt[vi],oof2[vi]):.5f} iters={m.best_iteration_}")
m2_oof_auc = roc_auc_score(yt, oof2)
print(f"  M2 OOF: {m2_oof_auc:.6f}")

best_auc, bw = 0., [0.5, 0.5]
for w1 in np.arange(0., 1.01, 0.05):
    auc = roc_auc_score(yt, w1*oof1 + (1-w1)*oof2)
    if auc > best_auc: best_auc = auc; bw = [w1, 1-w1]
oof_final = bw[0]*oof1 + bw[1]*oof2
tpred     = bw[0]*tst1 + bw[1]*tst2
print(f"\n  Blend: M1={bw[0]:.2f} M2={bw[1]:.2f} | OOF AUC={best_auc:.6f}")

prec_v, rec_v, pr_thr = precision_recall_curve(yt, oof_final)
pr_f1 = 2*prec_v[:-1]*rec_v[:-1] / (prec_v[:-1]+rec_v[:-1]+1e-9)
best_idx = int(np.argmax(pr_f1))
bth = float(pr_thr[best_idx]); bf1 = float(pr_f1[best_idx])
yb = (oof_final>=bth).astype(int)
print(f"  F1={bf1:.6f} @ {bth:.4f} | P={precision_score(yt,yb):.4f} | R={recall_score(yt,yb):.4f}")

# ══════════════════════════════════════════════════════════════════════════
# [9/9] EXPORT — full-data refit + serialize artifacts
# ══════════════════════════════════════════════════════════════════════════
print(f"\n[9/9] Exporting HAWKEYE artifacts to {ARTIFACTS}/ ...")

# ---- Refit on FULL training data (no CV) for production serving ----------
print("  Full-data refit M1...")
best_m1_iter = int(np.median(m1_iters))
m1_full = lgb.LGBMClassifier(**{**p1, 'n_estimators': best_m1_iter})
m1_full.fit(X1t, yt, sample_weight=weights)
m1_full.booster_.save_model(f'{ARTIFACTS}/lgb_model_m1_full.txt')

print("  Full-data refit M2...")
best_m2_iter = int(np.median(m2_iters))
m2_full = lgb.LGBMClassifier(**{**p2, 'n_estimators': best_m2_iter})
m2_full.fit(Xt, yt)
m2_full.booster_.save_model(f'{ARTIFACTS}/lgb_model_m2_full.txt')

# ---- feature_config.json -------------------------------------------------
feature_config = {
    'version': VERSION,
    'feat_cols':       feat_cols,        # full feature list, used by M2
    'feat_clean':      feat_clean,       # adversarial-cleaned, used by M1
    'excluded_adv':    rh,
    'blend_weights':   {'m1': float(bw[0]), 'm2': float(bw[1])},
    'threshold':       float(bth),
    'best_m1_iter':    best_m1_iter,
    'best_m2_iter':    best_m2_iter,
    'n_features_full':  len(feat_cols),
    'n_features_clean': len(feat_clean),
}
with open(f'{ARTIFACTS}/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print(f"  feature_config.json saved ({len(feat_cols)} feats, {len(feat_clean)} clean)")

# ---- feature_stats.json (for synthesis & input validation) ---------------
F_train = F.loc[tm, feat_cols].copy()
stats = {}
for c in feat_cols:
    s = F_train[c]
    stats[c] = dict(
        mean=float(np.nanmean(s)) if s.notna().any() else 0.,
        std =float(np.nanstd(s))  if s.notna().any() else 1.,
        min =float(np.nanmin(s))  if s.notna().any() else 0.,
        max =float(np.nanmax(s))  if s.notna().any() else 1.,
        p50 =float(np.nanpercentile(s, 50)) if s.notna().any() else 0.,
        p95 =float(np.nanpercentile(s, 95)) if s.notna().any() else 1.,
        nan_rate=float(s.isna().mean()))
with open(f'{ARTIFACTS}/feature_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
print(f"  feature_stats.json saved")

# ---- account_feature_matrix.parquet --------------------------------------
F_export = F.copy()
F_export['oof_proba'] = np.nan
F_export.loc[tm, 'oof_proba'] = oof_final
F_export.loc[~tm,'oof_proba'] = tpred
F_export.to_parquet(f'{ARTIFACTS}/account_feature_matrix.parquet', index=False)
print(f"  account_feature_matrix.parquet saved ({F_export.shape})")

# ---- oof_predictions.parquet --------------------------------------------
oof_df = pd.DataFrame({
    'account_id': F.loc[tm, 'account_id'].values,
    'is_mule':    yt,
    'oof_m1':     oof1,
    'oof_m2':     oof2,
    'oof_blend':  oof_final,
    'pred':       (oof_final>=bth).astype(int),
})
oof_df.to_parquet(f'{ARTIFACTS}/oof_predictions.parquet', index=False)

# ---- train_metadata.json ------------------------------------------------
metadata = {
    'version': VERSION,
    'oof_auc_m1':    float(m1_oof_auc),
    'oof_auc_m2':    float(m2_oof_auc),
    'oof_auc_blend': float(best_auc),
    'oof_f1':        float(bf1),
    'threshold':     float(bth),
    'precision':     float(precision_score(yt, yb)),
    'recall':        float(recall_score(yt, yb)),
    'n_train':       int(tm.sum()),
    'n_test':        int((~tm).sum()),
    'n_mules':       int(yt.sum()),
    'mule_rate':     float(yt.mean()),
    'n_features':    len(feat_cols),
    'n_features_clean': len(feat_clean),
    'training_time_seconds': time.time()-t0,
}
with open(f'{ARTIFACTS}/train_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  train_metadata.json saved")
print(f"\n  Performance: AUC={best_auc:.6f} | F1={bf1:.6f} | "
      f"P={metadata['precision']:.4f} | R={metadata['recall']:.4f}")

# ---- Original Kaggle submission (sanity) --------------------------------
tmax_map  = dict(zip(ts_df['account_id'], ts_df['tmax']))
tmin_map  = dict(zip(ts_df['account_id'], ts_df['tmin']))
score_map = dict(zip(tids, tpred))
windows = {}
for aid in tids:
    if score_map.get(aid, 0.) < 0.005: continue
    t_s = tmin_map.get(aid); t_e = tmax_map.get(aid)
    if t_s is None or t_e is None: continue
    try:
        t_s = pd.Timestamp(t_s); t_e = pd.Timestamp(t_e)
    except: continue
    if pd.isna(t_s) or pd.isna(t_e): continue
    if t_e <= t_s: t_e = t_s + pd.Timedelta(days=30)
    windows[aid] = {'s': t_s.isoformat(), 'e': t_e.isoformat()}
sub = pd.DataFrame({'account_id': tids, 'is_mule': tpred})
sub['suspicious_start'] = sub['account_id'].map(lambda x: windows.get(x,{}).get('s',''))
sub['suspicious_end']   = sub['account_id'].map(lambda x: windows.get(x,{}).get('e',''))
sub.to_csv(f'{ARTIFACTS}/submission.csv', index=False)
sub.to_csv(f'{OUTPUT}/submission_{VERSION}.csv', index=False)
print(f"  submission.csv saved")

print(f"\n{'='*70}")
print(f"  EXPORT COMPLETE — artifacts in {ARTIFACTS}/")
print(f"  Total time: {time.time()-t0:.0f}s")
print(f"{'='*70}")
print("\nFiles to download for HAWKEYE:")
for f in sorted(os.listdir(ARTIFACTS)):
    sz = os.path.getsize(f'{ARTIFACTS}/{f}')/1024
    print(f"  {f:<40} {sz:>10.1f} KB")

VERSION: HAWKEYE_export_v1
Train: 96,091 | Test: 64,062 | Mules: 2,683

[1/9] Main transaction loop (396 parts)...


Txn:   0%|          | 0/396 [00:00<?, ?it/s]

  Done 2236s | Edges: 3,254,788 | Stats: 160,153

[2/9] Loading tadd (balance + IP) via range-based matching...


tadd:   0%|          | 0/311 [00:00<?, ?it/s]

  Covered accounts: 160,153 | Mule cov: 2,683/2,683
  IP enabled: True | tadd pass: 4021s

[3/9] Graph features...
  Graph: (160153, 14)

[4/9] Burstiness...

[6/9] Static features...

[7/9] Merging...
  Feature matrix: (160153, 149)

[8/9] Training...
  Excluding 41: ['bal_range', 'b_burst_frac', 'age_d', 'bal_std', 'branch_asset_size', 'span', 'tpb', 'loan_sum', 'pmed', 'b_mom_max_jump', 'atb', 'nfr', 'p_mbd', 'kyc_d', 'psml', 'bal_cv', 'quarterly_avg_balance', 'bal_mean', 'b_peak_recency', 'b_cp_spike', 'age', 'b_spike_ratio', 'monthly_avg_balance', 'avg_balance', 'product_code', 'cdr', 'bal_max', 'fan_asymm', 'bal_min', 'branch_turnover', 'cc_sum', 'pbiz', 'pmic', 'pngt', 'bqd', 'branch_employee_count', 'pwkd', 'sa_sum', 'rel_y', 'bmd', 'fan_ratio']
  Noise detection pass...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[3]	valid_0's binary_logloss: 0.0921694
Training until validation scores don't improve for 50 rounds
Early stoppin

In [ ]:
#This performance metrics shown in above cell is inaccuarte , when tested on private and public leaderboard of Reserve Bank Innovation Hub , that is mentioned and used in our code already #4 rank 